In [ ]:
"""
1450 originals -> 2900 total samples.

Output:

    TrainDBAugmented/
    ├── images/          image_001.png, image_001_hflip.png, ...
    ├── points/          image_001.png, image_001_hflip.png, ...
    ├── pseudo/          image_001.png, image_001_hflip.png, ...
    └── image_names.txt  list of ALL filenames in the final dataset 
"""

import shutil
from pathlib import Path
from PIL import Image

# ---------------- CONFIG: edit these ----------------
ROOT = Path("/home/ubuntu/Downloads/Shivanshu Code/TapPolyPP/TrainDBNew/TrainDB/TrainDB")
IMAGE_DIR = ROOT / "image"
POINT_DIR = ROOT / "point"
PSEUDO_DIR = ROOT / "pseudo"

OUT_ROOT = Path("/home/ubuntu/Downloads/Shivanshu Code/TapPolyPP/TrainDBAugmented")
OUT_IMAGE_DIR = OUT_ROOT / "image"
OUT_POINT_DIR = OUT_ROOT / "point"
OUT_PSEUDO_DIR = OUT_ROOT / "pseudo"
NAMES_FILE = OUT_ROOT / "image_names.txt"

FLIP_SUFFIX = "_hflip"
# ------------------------------------------------------


def copy_original(path: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / path.name
    shutil.copy2(path, out_path)
    return out_path


def flip_and_save(path: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    img = Image.open(path)
    flipped = img.transpose(Image.FLIP_LEFT_RIGHT)
    out_path = out_dir / f"{path.stem}{FLIP_SUFFIX}{path.suffix}"
    flipped.save(out_path)
    return out_path


def main():
    image_files = sorted(IMAGE_DIR.glob("*.png"))
    if not image_files:
        print(f"No images found in {IMAGE_DIR}")
        return

    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    saved_names = []

    n_pairs = 0
    for img_path in image_files:
        stem = img_path.stem
        point_path = POINT_DIR / f"{stem}.png"
        pseudo_path = PSEUDO_DIR / f"{stem}.png"

        if not point_path.exists():
            print(f"[skip] no point map found for {stem}")
            continue
        if not pseudo_path.exists():
            print(f"[skip] no pseudo label found for {stem}")
            continue

        # --- copy originals as-is ---
        copy_original(img_path, OUT_IMAGE_DIR)
        copy_original(point_path, OUT_POINT_DIR)
        copy_original(pseudo_path, OUT_PSEUDO_DIR)
        saved_names.append(img_path.name)

        # --- flipped versions ---
        flip_and_save(img_path, OUT_IMAGE_DIR)
        flip_and_save(point_path, OUT_POINT_DIR)
        flip_and_save(pseudo_path, OUT_PSEUDO_DIR)
        saved_names.append(f"{stem}{FLIP_SUFFIX}{img_path.suffix}")

        n_pairs += 1
        print(f"[ok] {stem}  (+ {stem}{FLIP_SUFFIX})")

    with open(NAMES_FILE, "w") as f:
        f.write("\n".join(saved_names) + "\n")

    print(f"\nDone. {n_pairs} originals -> {len(saved_names)} total samples saved to {OUT_ROOT}")
    print(f"Image name list written to {NAMES_FILE}")


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path

# ---------------- CONFIG ----------------
IMAGE_DIR = Path("/home/ubuntu/Downloads/Shivanshu Code/TapPolyPP/TrainDBAugmented/image")
OUT_FILE = Path("/home/ubuntu/Downloads/Shivanshu Code/TapPolyPP/TrainDBAugmented/train.txt")
# -----------------------------------------


def main():
    image_files = sorted(IMAGE_DIR.glob("*.png"), key=lambda p: p.name)

    if not image_files:
        print(f"No images found in {IMAGE_DIR}")
        return

    with open(OUT_FILE, "w") as f:
        for p in image_files:
            f.write(p.stem + "\n")

    print(f"Wrote {len(image_files)} filenames to {OUT_FILE}")


if __name__ == "__main__":
    main()

In [ ]:
# =============================================================================
# Tap2Mask-Polyp (SAM ViT encoder): prompt-free polyp segmentation network.
# Partial release: interface only. Full implementation will be released
# upon paper acceptance.
# =============================================================================

import torch
import torch.nn as nn


class Tap2MaskPolyp(nn.Module):
    """
    Tap2Mask-Polyp: SAM ViT encoder -> multi-level fusion neck -> segmentation heads.

    Input : img_sam  (B, 3, 1024, 1024), SAM-preprocessed image
    Output: eval  -> main_logits (B, 1, H, W)
            train -> (main_logits, aux1_logits, aux2_logits)
    """

    def __init__(self, sam_model, cfg):
        super().__init__()
        # Private components live in a module that is not part of this release.
        from tap2mask_private import (
            SamViTMultiLevelBackbone, CrossScaleAttentionNeck,
            DecoderHead, FeatureDropout, FeatureNoise,
        )

        self.backbone = SamViTMultiLevelBackbone(sam_model, **cfg["backbone"])
        self.neck = CrossScaleAttentionNeck(
            in_channels=self.backbone.tap_channels, **cfg["neck"])

        ch, size = cfg["neck"]["out_channels"], cfg["out_size"]
        self.main_head = DecoderHead(ch, out_size=size)
        self.aux1_head = DecoderHead(ch, out_size=size)
        self.aux2_head = DecoderHead(ch, out_size=size)

        self.aux1_perturb = FeatureDropout(**cfg["aux1"])
        self.aux2_perturb = FeatureNoise(**cfg["aux2"])

    def forward(self, img_sam):
        fused = self.neck(self.backbone(img_sam))
        main_logits = self.main_head(fused)

        if not self.training:
            return main_logits

        return (main_logits,
                self.aux1_head(self.aux1_perturb(fused)),
                self.aux2_head(self.aux2_perturb(fused)))


def build_tap2mask_polyp_sam(sam_checkpoint, cfg, model_type="vit_b"):
    from segment_anything import sam_model_registry
    sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
    return Tap2MaskPolyp(sam, cfg)

In [ ]:
# =============================================================================
# Tap2Mask-Polyp (DINOv2 ViT encoder): prompt-free polyp segmentation network.
# Partial release: interface only. Full implementation will be released upon paper acceptance.
# =============================================================================

import torch
import torch.nn as nn


class Tap2MaskPolyp(nn.Module):
    """
    Tap2Mask-Polyp: DINOv2 ViT encoder -> multi-level fusion neck -> segmentation heads.

    Input : img (B, 3, H, W), preprocessed image; H and W are set by
            cfg["backbone"]["encoder_input_size"]
    Output: eval  -> main_logits (B, 1, out_size, out_size)
            train -> (main_logits, aux1_logits, aux2_logits)
    """

    def __init__(self, cfg):
        super().__init__()
        from tap2mask_private import (
            Dinov2MultiLevelBackbone, FeatureFusionNeck,
            TransformerDecoderHead, DecoderHead,
            FeatureDropout, FeatureNoise,
        )

        self.backbone = Dinov2MultiLevelBackbone(**cfg["backbone"])
        self.neck = FeatureFusionNeck(
            in_channels=self.backbone.tap_channels,
            n_levels=self.backbone.n_levels,
            **cfg["neck"])

        ch, size = cfg["neck"]["out_channels"], cfg["out_size"]
        self.main_head = TransformerDecoderHead(ch, out_size=size, **cfg["main_head"])
        self.aux1_head = DecoderHead(ch, out_size=size)
        self.aux2_head = DecoderHead(ch, out_size=size)

        self.aux1_perturb = FeatureDropout(**cfg["aux1"])
        self.aux2_perturb = FeatureNoise(**cfg["aux2"])

    def forward(self, img):
        fused = self.neck(self.backbone(img))
        main_logits = self.main_head(fused)

        if not self.training:
            return main_logits

        return (main_logits,
                self.aux1_head(self.aux1_perturb(fused)),
                self.aux2_head(self.aux2_perturb(fused)))


def build_tap2mask_polyp_dinov2(cfg):
    return Tap2MaskPolyp(cfg)

In [ ]:
# coding=utf-8
"""
Test script for the trained Tap2Mask-Polyp model.
"""

import os
import time
import cv2
import numpy as np
import torch
import torch.nn.functional as F

import py_sod_metrics


TEST_ROOT = "/home/ubuntu/Downloads/Shivanshu Code/TapPolyPP/TestDB/TestDB"
DATASETS = ["CVC-300", "CVC-ClinicDB", "CVC-ColonDB", "Kvasir", "ETIS-LaribPolypDB"]

IMAGE_SUBDIR = "images"
MASK_SUBDIR = "masks"

# Checkpoint saved by train_tap2mask_polyp.py
CHECKPOINT_PATH = "/home/ubuntu/Downloads/Shivanshu Code/TapPolyPP/working/Models_TAPPOLYP-E2E-GD-ORIGINAL-OT/model-100.pt"

SAM_CHECKPOINT = "/home/ubuntu/Downloads/Shivanshu Code/sam_vit_b_01ec64.pth"
MODEL_TYPE = "vit_b"

INPUT_SIZE = 320  # matches out_size used in training / model construction


def Dice(pred, mask):
    inter = (pred * mask).sum()
    union = (pred + mask).sum()
    return (2 * inter + 1e-10) / (union + 1e-10)


def IoU(pred, mask):
    inter = (pred * mask).sum()
    union = (pred + mask).sum() - inter
    return (inter + 1e-10) / (union + 1e-10)


def preprocess(image):
    """image: HxWx3 float32 RGB, raw 0-255 range. No mean/std subtraction --
    the model does its own SAM-style normalization internally. Returns
    raw-pixel CHW tensor resized to INPUT_SIZE, plus original (H, W)."""
    H, W = image.shape[:2]
    resized = cv2.resize(image, (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_LINEAR)
    chw = resized.transpose(2, 0, 1)  # HWC -> CHW
    return torch.from_numpy(chw).float(), (H, W)


def load_mask(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
    mask = (mask > 127).astype(np.float32)  # binarize GT mask
    return mask


def evaluate_dataset(net, dataset_name, device):
    dataset_dir = os.path.join(TEST_ROOT, dataset_name)
    image_dir = os.path.join(dataset_dir, IMAGE_SUBDIR)
    mask_dir = os.path.join(dataset_dir, MASK_SUBDIR)

    if not os.path.exists(image_dir) or not os.path.exists(mask_dir):
        print(f"[SKIP] {dataset_name}: expected {image_dir} and {mask_dir} - check folder names")
        return

    filenames = sorted([f for f in os.listdir(image_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    if len(filenames) == 0:
        print(f"[SKIP] {dataset_name}: no images found in {image_dir}")
        return

    dice_total, iou_total, mae_total, cnt = 0.0, 0.0, 0.0, 0
    cost_time = 0.0

    sm_metric = py_sod_metrics.Smeasure()
    wfm_metric = py_sod_metrics.WeightedFmeasure()

    with torch.no_grad():
        for fname in filenames:
            image_path = os.path.join(image_dir, fname)
            mask_path = os.path.join(mask_dir, fname)
            if not os.path.exists(mask_path):
                print(f"  [WARN] no matching mask for {fname}, skipping")
                continue

            image_bgr = cv2.imread(image_path).astype(np.float32)
            image_rgb = image_bgr[:, :, ::-1]
            image_tensor, (H, W) = preprocess(image_rgb)
            image_tensor = image_tensor.unsqueeze(0).to(device)

            mask = load_mask(mask_path)
            mask_tensor = torch.from_numpy(mask).float().to(device)

            start = time.time()
            main_logits = net(image_tensor)             # eval() -> ONE tensor
            pred = torch.sigmoid(main_logits[:, 1:2])    # (1, 1, INPUT_SIZE, INPUT_SIZE)
            torch.cuda.synchronize() if device == 'cuda' else None
            cost_time += time.time() - start

            pred = F.interpolate(pred, size=(H, W), mode='bilinear', align_corners=False)
            pred = pred[0, 0]
            pred_bin = (pred > 0.5).float()

            dice_total += Dice(pred_bin, mask_tensor).item()
            iou_total += IoU(pred_bin, mask_tensor).item()
            mae_total += (pred - mask_tensor).abs().mean().item()
            cnt += 1

            pred_255 = (pred.cpu().numpy() * 255).astype(np.uint8)
            mask_255 = (mask * 255).astype(np.uint8)
            sm_metric.step(pred=pred_255, gt=mask_255)
            wfm_metric.step(pred=pred_255, gt=mask_255)

    if cnt == 0:
        print(f"[SKIP] {dataset_name}: no valid image/mask pairs")
        return

    fps = cnt / cost_time if cost_time > 0 else float('nan')
    sm = sm_metric.get_results()["sm"]
    wfm = wfm_metric.get_results()["wfm"]
    print(f"{dataset_name:20s} | Dice={dice_total/cnt:.4f} | IoU={iou_total/cnt:.4f} | "
          f"Sm={sm:.4f} | wFm={wfm:.4f} | MAE={mae_total/cnt:.4f} | n={cnt} | fps={fps:.2f}")


def main():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    net = build_tap2mask_polyp_sam(
        sam_checkpoint=SAM_CHECKPOINT,
        model_type=MODEL_TYPE,
        out_size=INPUT_SIZE,
        tap_blocks=(3, 6, 9, 11),
        n_unfreeze=2,
    ).to(device)
    state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
    net.load_state_dict(state_dict)
    net.eval()

    print(f"Loaded checkpoint: {CHECKPOINT_PATH}")
    print(f"Device: {device}\n")

    for dataset_name in DATASETS:
        evaluate_dataset(net, dataset_name, device)


if __name__ == "__main__":
    main()

In [ ]:
import os
import random
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# =====================================================
# CONFIG
# =====================================================
TEST_ROOT = "/home/ubuntu/Downloads/Shivanshu Code/TapPolyPP/TestDB/TestDB"

DATASETS = [
    "CVC-300",
    "CVC-ClinicDB",
    "CVC-ColonDB",
    "Kvasir",
    "ETIS-LaribPolypDB"
]

IMAGE_SUBDIR = "images"
MASK_SUBDIR = "masks"

CHECKPOINT_PATH = "/home/ubuntu/Downloads/Shivanshu Code/TapPolyPP/working/Models_TAPPOLYP-E2E-GD-ORIGINAL-OT/model-150.pt"

SAM_CHECKPOINT = "/home/ubuntu/Downloads/Shivanshu Code/sam_vit_b_01ec64.pth"

MODEL_TYPE = "vit_b"
INPUT_SIZE = 320

SAVE_ROOT = "/home/ubuntu/Downloads/Shivanshu Code/TapPolyPP/working/visualization_results/TapPolyp"
NUM_SAMPLES = 4


# =====================================================
# PREPROCESS
# =====================================================
def preprocess(image):
    H, W = image.shape[:2]
    resized = cv2.resize(image, (INPUT_SIZE, INPUT_SIZE))
    tensor = torch.from_numpy(resized.transpose(2,0,1)).float()
    return tensor,(H,W)


def load_mask(path):
    mask = cv2.imread(path,0)
    return (mask>127).astype(np.uint8)


# =====================================================
# OVERLAY
# =====================================================
def create_overlay(image, mask, alpha=0.6):

    image = image.astype(np.float32) / 255.0
    overlay = image.copy()

    # Bright green
    green = np.array([0.0, 1.0, 0.0])

    for c in range(3):
        overlay[:, :, c] = np.where(
            mask > 0,
            (1 - alpha) * image[:, :, c] + alpha * green[c],
            image[:, :, c]
        )

    overlay = np.clip(overlay, 0, 1)
    return (overlay * 255).astype(np.uint8)


# =====================================================
# VISUALIZE DATASET
# =====================================================
NUM_SAMPLES = 2  # just for display now, not saving


# =====================================================
# VISUALIZE DATASET
# =====================================================
def visualize_dataset(net, dataset_name, device):

    image_dir = os.path.join(TEST_ROOT, dataset_name, IMAGE_SUBDIR)
    mask_dir = os.path.join(TEST_ROOT, dataset_name, MASK_SUBDIR)

    files = sorted(os.listdir(image_dir))

    # pick which ones get plotted (subset of what gets saved)
    display_set = set(random.sample(files, min(NUM_SAMPLES, len(files))))

    os.makedirs(os.path.join(SAVE_ROOT, dataset_name), exist_ok=True)

    display_rows = []  # will hold (rgb, gt_overlay, pred_overlay) for plotting

    with torch.no_grad():

        for fname in files:  # <-- loop over ALL files now, not just chosen

            img_path = os.path.join(image_dir, fname)
            mask_path = os.path.join(mask_dir, fname)

            bgr = cv2.imread(img_path)
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

            gt = load_mask(mask_path)

            tensor, (H, W) = preprocess(rgb)
            tensor = tensor.unsqueeze(0).to(device)

            logits = net(tensor)

            pred = torch.sigmoid(logits[:, 1:2])

            pred = F.interpolate(pred,
                                  size=(H, W),
                                  mode="bilinear",
                                  align_corners=False)

            pred = (pred[0, 0].cpu().numpy() > 0.5).astype(np.uint8)

            gt_overlay = create_overlay(rgb, gt)
            pred_overlay = create_overlay(rgb, pred)

            sample = os.path.splitext(fname)[0]

            save_dir = os.path.join(SAVE_ROOT, dataset_name, sample)
            os.makedirs(save_dir, exist_ok=True)

            cv2.imwrite(os.path.join(save_dir, "input.png"),
                        cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

            cv2.imwrite(os.path.join(save_dir, "gt.png"), gt * 255)

            cv2.imwrite(os.path.join(save_dir, "prediction.png"), pred * 255)

            cv2.imwrite(os.path.join(save_dir, "gt_overlay.png"),
                        cv2.cvtColor(gt_overlay, cv2.COLOR_RGB2BGR))

            cv2.imwrite(os.path.join(save_dir, "prediction_overlay.png"),
                        cv2.cvtColor(pred_overlay, cv2.COLOR_RGB2BGR))

            # stash for plotting only if this file was picked for display
            if fname in display_set:
                display_rows.append((rgb, gt_overlay, pred_overlay))

    # ---- plot only the chosen samples ----
    fig, axs = plt.subplots(len(display_rows), 3,
                             figsize=(12, 4 * len(display_rows)))

    if len(display_rows) == 1:
        axs = np.expand_dims(axs, 0)

    for row, (rgb, gt_overlay, pred_overlay) in enumerate(display_rows):
        axs[row, 0].imshow(rgb)
        axs[row, 0].set_title("Input")

        axs[row, 1].imshow(gt_overlay)
        axs[row, 1].set_title("Ground Truth")

        axs[row, 2].imshow(pred_overlay)
        axs[row, 2].set_title("Prediction")

        for j in range(3):
            axs[row, j].axis("off")

    plt.suptitle(dataset_name, fontsize=16)
    plt.tight_layout()
    plt.show()

# =====================================================
# MAIN
# =====================================================
device="cuda" if torch.cuda.is_available() else "cpu"

net=build_tap2mask_polyp_sam(
    sam_checkpoint=SAM_CHECKPOINT,
    model_type=MODEL_TYPE,
    out_size=INPUT_SIZE,
    tap_blocks=(3,6,9,11),
    n_unfreeze=2,
).to(device)

state_dict=torch.load(CHECKPOINT_PATH,
                      map_location=device)

net.load_state_dict(state_dict)
net.eval()

print("Model Loaded.\n")

for dataset in DATASETS:
    print(f"Processing {dataset}")
    visualize_dataset(net,dataset,device)

print("\nSaved to:",SAVE_ROOT)

In [ ]:
import os
import random
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# =====================================================
# CONFIG
# =====================================================
TEST_ROOT = "/home/ubuntu/Downloads/Shivanshu Code/TapPolyPP/TestDB/TestDB"

DATASETS = [
    "CVC-300",
    "CVC-ClinicDB",
    "CVC-ColonDB",
    "Kvasir",
    "ETIS-LaribPolypDB"
]

IMAGE_SUBDIR = "images"
MASK_SUBDIR = "masks"

CHECKPOINT_PATH = "/home/ubuntu/Downloads/Shivanshu Code/TapPolyPP/working/Models_TAPPOLYP-E2E-GD-ORIGINAL-OT-DINOV2/model-150.pt"

DINOV2_MODEL_NAME = "dinov2_vitb14"

ENCODER_INPUT_SIZE = 896 

INPUT_SIZE = 320

SAVE_ROOT = "/home/ubuntu/Downloads/Shivanshu Code/TapPolyPP/working/visualization_results/Tap2Mask-Polyp-ours-dINO"
NUM_SAMPLES = 4


# =====================================================
# PREPROCESS
# =====================================================
def preprocess(image):
    H, W = image.shape[:2]
    resized = cv2.resize(image, (INPUT_SIZE, INPUT_SIZE))
    tensor = torch.from_numpy(resized.transpose(2,0,1)).float()
    return tensor,(H,W)


def load_mask(path):
    mask = cv2.imread(path,0)
    return (mask>127).astype(np.uint8)


# =====================================================
# OVERLAY
# =====================================================
def create_overlay(image, mask, alpha=0.6):

    image = image.astype(np.float32) / 255.0
    overlay = image.copy()

    # Bright green
    green = np.array([0.0, 1.0, 0.0])

    for c in range(3):
        overlay[:, :, c] = np.where(
            mask > 0,
            (1 - alpha) * image[:, :, c] + alpha * green[c],
            image[:, :, c]
        )

    overlay = np.clip(overlay, 0, 1)
    return (overlay * 255).astype(np.uint8)


# =====================================================
# VISUALIZE DATASET
# =====================================================
NUM_SAMPLES = 2  # just for display now, not saving


# =====================================================
# VISUALIZE DATASET
# =====================================================
def visualize_dataset(net, dataset_name, device):

    image_dir = os.path.join(TEST_ROOT, dataset_name, IMAGE_SUBDIR)
    mask_dir = os.path.join(TEST_ROOT, dataset_name, MASK_SUBDIR)

    files = sorted(os.listdir(image_dir))

    # pick which ones get plotted (subset of what gets saved)
    display_set = set(random.sample(files, min(NUM_SAMPLES, len(files))))

    os.makedirs(os.path.join(SAVE_ROOT, dataset_name), exist_ok=True)

    display_rows = []  # will hold (rgb, gt_overlay, pred_overlay) for plotting

    with torch.no_grad():

        for fname in files:  # <-- loop over ALL files now, not just chosen

            img_path = os.path.join(image_dir, fname)
            mask_path = os.path.join(mask_dir, fname)

            bgr = cv2.imread(img_path)
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

            gt = load_mask(mask_path)

            tensor, (H, W) = preprocess(rgb)
            tensor = tensor.unsqueeze(0).to(device)

            logits = net(tensor)

            pred = torch.sigmoid(logits[:, 1:2])

            pred = F.interpolate(pred,
                                  size=(H, W),
                                  mode="bilinear",
                                  align_corners=False)

            pred = (pred[0, 0].cpu().numpy() > 0.5).astype(np.uint8)

            gt_overlay = create_overlay(rgb, gt)
            pred_overlay = create_overlay(rgb, pred)

            sample = os.path.splitext(fname)[0]

            save_dir = os.path.join(SAVE_ROOT, dataset_name, sample)
            os.makedirs(save_dir, exist_ok=True)

            cv2.imwrite(os.path.join(save_dir, "input.png"),
                        cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

            cv2.imwrite(os.path.join(save_dir, "gt.png"), gt * 255)

            cv2.imwrite(os.path.join(save_dir, "prediction.png"), pred * 255)

            cv2.imwrite(os.path.join(save_dir, "gt_overlay.png"),
                        cv2.cvtColor(gt_overlay, cv2.COLOR_RGB2BGR))

            cv2.imwrite(os.path.join(save_dir, "prediction_overlay.png"),
                        cv2.cvtColor(pred_overlay, cv2.COLOR_RGB2BGR))

            # stash for plotting only if this file was picked for display
            if fname in display_set:
                display_rows.append((rgb, gt_overlay, pred_overlay))

    # ---- plot only the chosen samples ----
    fig, axs = plt.subplots(len(display_rows), 3,
                             figsize=(12, 4 * len(display_rows)))

    if len(display_rows) == 1:
        axs = np.expand_dims(axs, 0)

    for row, (rgb, gt_overlay, pred_overlay) in enumerate(display_rows):
        axs[row, 0].imshow(rgb)
        axs[row, 0].set_title("Input")

        axs[row, 1].imshow(gt_overlay)
        axs[row, 1].set_title("Ground Truth")

        axs[row, 2].imshow(pred_overlay)
        axs[row, 2].set_title("Prediction")

        for j in range(3):
            axs[row, j].axis("off")

    plt.suptitle(dataset_name, fontsize=16)
    plt.tight_layout()
    plt.show()

# =====================================================
# MAIN
# =====================================================
device="cuda" if torch.cuda.is_available() else "cpu"

net = build_tap2mask_polyp_dinov2(
    model_name=DINOV2_MODEL_NAME,
    out_size=INPUT_SIZE,
    tap_blocks=(3, 6, 9, 11),
    n_unfreeze=2,
    encoder_input_size=ENCODER_INPUT_SIZE,
).to(device)

state_dict=torch.load(CHECKPOINT_PATH,
                      map_location=device)

net.load_state_dict(state_dict)
net.eval()

print("Model Loaded.\n")

for dataset in DATASETS:
    print(f"Processing {dataset}")
    visualize_dataset(net,dataset,device)

print("\nSaved to:",SAVE_ROOT)